# MCP Tools — Test Notebook

Verifies all MCP tools used by AACAgent.

## Rules
- **No `_DatasetCache` imports**: only MCP tool APIs are used.
- Online/offline consistency is tested by monkey-patching `USE_LOCAL_DATASETS`
  in the tool module (not via direct cache access).
- Tests run against the local dataset by default (`USE_LOCAL_DATASETS=True`).

## Sections
1. Setup & imports
2. `list_keywords`
3. `search_pictograms`
4. `get_pictogram_metadata`
5. `search_pictograms_by_synset`
6. `get_time`
7. `get_schedule`
8. Online / offline consistency (monkey-patch)
9. `resolve_concept`


In [26]:
import sys, os
from pathlib import Path

# Resolve project root and app/src/ — works whether run from test/ or project root
HERE = Path('.').resolve()
# Detect that we are inside test/ by checking for the notebook file
if (HERE / 'tools_test.ipynb').exists():
    PROJECT_ROOT = HERE.parent          # test/ → project root
else:
    PROJECT_ROOT = HERE                 # already at project root
SRC = PROJECT_ROOT / 'app' / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print('PROJECT_ROOT:', PROJECT_ROOT)
print('SRC:', SRC)


PROJECT_ROOT: /Users/pelle/Development/GitHub/aac-mcp-agent
SRC: /Users/pelle/Development/GitHub/aac-mcp-agent/app/src


In [27]:
from config import LANG, USE_LOCAL_DATASETS, DATASETS_DIR
from mcp_server.tools.arasaac import (
    list_keywords,
    search_pictograms,
    get_pictogram_metadata,
    search_pictograms_by_synset,
)
from mcp_server.tools.time_tool     import get_time
from mcp_server.tools.schedule_tool import get_schedule
import mcp_server.tools.arasaac as _arasaac_mod  # for monkey-patching

print(f'LANG={LANG!r}  USE_LOCAL_DATASETS={USE_LOCAL_DATASETS}')
print(f'DATASETS_DIR exists: {DATASETS_DIR.exists()}')


LANG='en'  USE_LOCAL_DATASETS=True
DATASETS_DIR exists: True


## 1. `list_keywords`


In [28]:
kw_result = list_keywords(lang=LANG)
assert isinstance(kw_result, dict), 'Expected dict'
assert 'keywords' in kw_result, 'Missing keywords key'
kws = kw_result['keywords']
assert isinstance(kws, list), 'keywords must be a list'
print(f'Total keywords: {len(kws):,}')
print(f'First 10: {kws[:10]}')


[05/12/26 10:39:26] INFO     list_keywords(lang='en'): 15757 keywords from local dataset.            ]8;id=7010135;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=7010136;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#316\316]8;;\

Total keywords: 15,757
First 10: ['!', '"', '#', '$', '%', '(', ')', '*', '+', ',']


In [29]:
# Build set for O(1) lookup — mirrors agent._load_kw_set()
kw_set = set(kws)
assert len(kw_set) > 10_000, f'Expected >10k keywords, got {len(kw_set)}'

# Known single-word keywords that must be present
required = ['water', 'eat', 'coat', 'bag', 'shoes']
missing  = [k for k in required if k not in kw_set]
assert not missing, f'Missing known keywords: {missing}'
print('All required single-word keywords present.')

# Multi-word entries must exist (see context §13)
multi = [k for k in kw_set if ' ' in k]
print(f'Multi-word keywords: {len(multi):,} ({len(multi)/len(kw_set)*100:.1f}%)')
print(f'Sample multi-word: {multi[:5]}')


All required single-word keywords present.
Multi-word keywords: 7,850 (49.8%)
Sample multi-word: ['interact with', 'shared dialog', 'switch over', 'cinnamon bun', 'flash memory']


## 2. `search_pictograms`


In [30]:
# Basic search — single exact keyword
res = search_pictograms(keyword='water', lang=LANG, max_results=5)
assert isinstance(res, dict)
assert 'results' in res
pics = res['results']
assert isinstance(pics, list)
assert len(pics) > 0, 'Expected results for "water"'
print(f'search_pictograms("water") → {len(pics)} results')
for p in pics:
    print(f'  id={p["id"]}  kws={[k["keyword"] for k in p.get("keywords", [])[:3]]}')


[05/12/26 10:39:27] INFO     search_pictograms('water', lang='en'): 5 results from local dataset.    ]8;id=7010141;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=7010142;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#227\227]8;;\

search_pictograms("water") → 5 results
  id=22082  kws=['water', 'irrigate']
  id=2248  kws=['water']
  id=24354  kws=['water', 'spray', 'irrigate']
  id=24355  kws=['water', 'irrigate', 'spray']
  id=2816  kws=['water']


In [31]:
# Each result must have the expected fields
required_fields = {'id', 'keywords', 'categories', 'synsets', 'tags', 'aac', 'violence', 'sex'}
for p in pics:
    missing = required_fields - set(p.keys())
    assert not missing, f'Pictogram id={p.get("id")} missing fields: {missing}'
print('All required fields present in search results.')


All required fields present in search results.


In [32]:
# max_results is respected
res2 = search_pictograms(keyword='eat', lang=LANG, max_results=2)
assert len(res2['results']) <= 2, 'max_results not respected'
print(f'max_results=2 → {len(res2["results"])} results (ok)')


                    INFO     search_pictograms('eat', lang='en'): 2 results from local dataset.      ]8;id=7010147;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=7010148;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#227\227]8;;\

max_results=2 → 2 results (ok)


In [33]:
# Multi-word keyword (verbatim from index)
if 'go out' in kw_set:
    res_mw = search_pictograms(keyword='go out', lang=LANG, max_results=5)
    assert len(res_mw['results']) > 0, '"go out" in index but no results'
    print(f'search_pictograms("go out") → {len(res_mw["results"])} results')
else:
    print('"go out" not in kw_set for this lang — skipping')


                    INFO     search_pictograms('go out', lang='en'): 4 results from local dataset.   ]8;id=7010153;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=7010154;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#227\227]8;;\

search_pictograms("go out") → 4 results


In [34]:
# Non-existent keyword → empty results (no exception)
res_empty = search_pictograms(keyword='__nonexistent_xyz__', lang=LANG)
assert res_empty['results'] == [], f'Expected empty list, got {res_empty["results"]}'
print('Non-existent keyword → empty list (ok, no exception)')


                    INFO     search_pictograms('__nonexistent_xyz__', lang='en'): 0 results from     ]8;id=7010159;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=7010160;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#227\227]8;;\
                             local dataset.                                                                        

Non-existent keyword → empty list (ok, no exception)


In [35]:
# No duplicate IDs in results
ids = [p['id'] for p in res['results']]
assert len(ids) == len(set(ids)), 'Duplicate IDs in search results'
print('No duplicate IDs in search results (ok)')


No duplicate IDs in search results (ok)


## 3. `get_pictogram_metadata`


In [36]:
# Known pictogram: water (id=2248)
WATER_ID = 2248
meta = get_pictogram_metadata(pictogram_id=WATER_ID, lang=LANG)
assert 'error' not in meta, f'Unexpected error: {meta}'
assert meta['id'] == WATER_ID
kws = [k['keyword'] for k in meta.get('keywords', [])]
assert 'water' in kws, f'"water" keyword not found in id={WATER_ID}: {kws}'
print(f'get_pictogram_metadata(id={WATER_ID})')
print(f'  keywords : {kws[:5]}')
print(f'  categories: {meta.get("categories", [])}')
print(f'  synsets  : {meta.get("synsets", [])}')
print(f'  aac      : {meta.get("aac")}')


                    INFO     get_pictogram_metadata(id=2248): OK from local dataset.                 ]8;id=7010165;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=7010166;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#274\274]8;;\

get_pictogram_metadata(id=2248)
  keywords : ['water']
  categories: ['beverage', 'mineral rich food']
  synsets  : ['07951744-n', '14869913-n']
  aac      : False


In [37]:
# All expected fields present
required_meta_fields = {'id', 'keywords', 'categories', 'synsets', 'tags',
                        'aac', 'violence', 'sex', 'image_url'}
missing_meta = required_meta_fields - set(meta.keys())
assert not missing_meta, f'Missing fields in metadata: {missing_meta}'
# image_url is '/api/images/{id}' when USE_LOCAL_DATASETS=True, else a CDN URL
assert meta['image_url'].startswith('http') or meta['image_url'].startswith('/api/images/'), \
    f'image_url unexpected format: {meta["image_url"]}'
print('All required metadata fields present.')


All required metadata fields present.


In [40]:
# Unknown ID → error dict, no exception
meta_bad = get_pictogram_metadata(pictogram_id=999_999_999, lang=LANG)
assert 'error' in meta_bad, f'Expected error dict for unknown ID, got: {meta_bad}'
print(f'Unknown ID → error dict (ok, no exception)  got: {meta_bad}')

[05/12/26 10:41:33] INFO     get_pictogram_metadata(id=999999999): not found on API.                 ]8;id=7010184;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=7010185;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#288\288]8;;\

Unknown ID → error dict (ok, no exception)  got: {'error': 'Pictogram id=999999999 not found on ARASAAC'}


In [41]:
# Roundtrip: search → take first ID → get_metadata
first_id = res['results'][0]['id']
meta_rt  = get_pictogram_metadata(pictogram_id=first_id, lang=LANG)
assert meta_rt.get('id') == first_id, f'Roundtrip id mismatch: {meta_rt.get("id")} != {first_id}'
print(f'Roundtrip search→metadata for id={first_id}: ok')


[05/12/26 10:41:40] INFO     get_pictogram_metadata(id=22082): OK from local dataset.                ]8;id=7010190;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=7010191;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#274\274]8;;\

Roundtrip search→metadata for id=22082: ok


## 4. `search_pictograms_by_synset`


In [42]:
# Known synset: water entity (WordNet 3.1)
WATER_SYNSET = '07951744-n'
syn_res = search_pictograms_by_synset(synset_id=WATER_SYNSET, lang=LANG)
assert isinstance(syn_res, dict)
assert 'results' in syn_res
syn_pics = syn_res['results']
print(f'search_pictograms_by_synset("{WATER_SYNSET}") → {len(syn_pics)} results')
for p in syn_pics[:3]:
    print(f'  id={p["id"]}  kws={[k["keyword"] for k in p.get("keywords", [])[:2]]}')


                    INFO     Loaded synset_index for lang='en' (8422 synsets).                 ]8;id=7010197;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/dataset_cache.py\dataset_cache.py]8;;\:]8;id=7010198;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/dataset_cache.py#119\119]8;;\

                    INFO     search_pictograms_by_synset(synset=07951744-n, lang=en): 3 results from ]8;id=7010204;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=7010205;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#372\372]8;;\
                             local dataset.                                                                        

search_pictograms_by_synset("07951744-n") → 3 results
  id=2248  kws=['water']
  id=32464  kws=['water']
  id=6889  kws=['water']


In [43]:
# Results must contain pictogram id=2248 (water) — it has this synset
synset_ids = {p['id'] for p in syn_pics}
assert WATER_ID in synset_ids, (
    f'Expected id={WATER_ID} in synset results for {WATER_SYNSET}, got: {synset_ids}'
)
print(f'id={WATER_ID} found in synset results (ok)')


id=2248 found in synset results (ok)


In [44]:
# Unknown synset → empty results, no exception
syn_empty = search_pictograms_by_synset(synset_id='00000000-x', lang=LANG)
assert syn_empty['results'] == [], f'Expected empty list, got {syn_empty["results"]}'
print('Unknown synset → empty list (ok, no exception)')


                    INFO     search_pictograms_by_synset(synset=00000000-x, lang=en): 0 results from ]8;id=7010210;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=7010211;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#372\372]8;;\
                             local dataset.                                                                        

Unknown synset → empty list (ok, no exception)


In [45]:
# Cross-check: take a pictogram from search_pictograms that has synsets,
# call search_by_synset → the original pictogram must appear in the results.
source_pic = next(
    (p for p in res['results'] if p.get('synsets')), None
)
if source_pic:
    test_synset = source_pic['synsets'][0]
    back_res    = search_pictograms_by_synset(synset_id=test_synset, lang=LANG)
    back_ids    = {p['id'] for p in back_res['results']}
    assert source_pic['id'] in back_ids, (
        f'Pictogram id={source_pic["id"]} not found via its own synset {test_synset}'
    )
    print(f'Cross-check synset {test_synset!r} → id={source_pic["id"]} found (ok)')
else:
    print('No synset-bearing pictogram in search results — cross-check skipped')


                    INFO     search_pictograms_by_synset(synset=00228662-v, lang=en): 6 results from ]8;id=7010216;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=7010217;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#372\372]8;;\
                             local dataset.                                                                        

Cross-check synset '00228662-v' → id=22082 found (ok)


## 5. `get_time`


In [46]:
from config import DAY_TIMES

time_result = get_time()
assert isinstance(time_result, dict), f'Expected dict, got {type(time_result)}'
assert 'current_dt' in time_result, 'Missing current_dt'
assert 'time_of_day' in time_result, 'Missing time_of_day'
print(f'get_time() → current_dt={time_result["current_dt"]!r}')
print(f'             time_of_day={time_result["time_of_day"]!r}')


get_time() → current_dt='2026-05-12T10:41:40.559426'
             time_of_day='morning'


In [47]:
# time_of_day must be one of the configured slots
assert time_result['time_of_day'] in DAY_TIMES, (
    f'time_of_day={time_result["time_of_day"]!r} not in DAY_TIMES={DAY_TIMES}'
)
# current_dt must be parseable as ISO 8601
from datetime import datetime
dt = datetime.fromisoformat(str(time_result['current_dt']))
print(f'current_dt parsed ok: {dt}  |  time_of_day valid: {time_result["time_of_day"]}')


current_dt parsed ok: 2026-05-12 10:41:40.559426  |  time_of_day valid: morning


## 6. `get_schedule`


In [48]:
schedule = get_schedule()
assert isinstance(schedule, list), f'Expected list, got {type(schedule)}'
print(f'get_schedule() → {len(schedule)} events')


[05/12/26 10:41:47] INFO     Apple CalDAV: 2 events for 2026-05-12                             ]8;id=7010224;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/schedule_tool.py\schedule_tool.py]8;;\:]8;id=7010225;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/schedule_tool.py#214\214]8;;\

get_schedule() → 2 events


In [49]:
# If events are returned, each must have required fields
required_event_fields = {'title', 'start_time'}
for ev in schedule:
    assert isinstance(ev, dict), f'Event must be dict, got {type(ev)}'
    missing_ev = required_event_fields - set(ev.keys())
    assert not missing_ev, f'Event missing fields: {missing_ev}  event={ev}'
print(f'All {len(schedule)} events have required fields (title, start_time)')
for ev in schedule[:3]:
    print(f'  {ev["start_time"]}  {ev["title"]!r}')


All 2 events have required fields (title, start_time)
  10:13:00  'breakfast with ice cream'
  21:30:00  'Allenamento CSI'


## 7. Online / offline consistency

Compares tool output with `USE_LOCAL_DATASETS=True` vs `False` by
monkey-patching the module-level variable in `mcp_server.tools.arasaac`.

**Rule:** only MCP tool APIs are used — `_DatasetCache` is never imported here.
The module-internal branching is exactly what we want to test end-to-end.

⚠ Requires network connectivity.  Cells are wrapped in try/except so they
degrade gracefully when running fully offline.


In [50]:
import contextlib

@contextlib.contextmanager
def _use_api():
    """Temporarily force the tool module to use the live API."""
    old = _arasaac_mod.USE_LOCAL_DATASETS
    _arasaac_mod.USE_LOCAL_DATASETS = False
    try:
        yield
    finally:
        _arasaac_mod.USE_LOCAL_DATASETS = old

print(f'Current module USE_LOCAL_DATASETS: {_arasaac_mod.USE_LOCAL_DATASETS}')
print('Monkey-patch context manager ready.')


Current module USE_LOCAL_DATASETS: True
Monkey-patch context manager ready.


In [51]:
# list_keywords: local vs API — keyword count should be close
try:
    local_kws = set(list_keywords(lang=LANG)['keywords'])
    with _use_api():
        api_kws = set(list_keywords(lang=LANG)['keywords'])

    jaccard = len(local_kws & api_kws) / len(local_kws | api_kws)
    print(f'list_keywords  local={len(local_kws):,}  api={len(api_kws):,}  jaccard={jaccard:.3f}')
    assert jaccard > 0.90, f'list_keywords local/API overlap too low: {jaccard:.3f}'
    print('  consistency ok')
except Exception as exc:
    print(f'  SKIP (network error): {exc}')


                    INFO     list_keywords(lang='en'): 15757 keywords from local dataset.            ]8;id=7010230;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=7010231;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#316\316]8;;\

                    INFO     list_keywords(lang='en'): local dataset unavailable, calling API.       ]8;id=7010237;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=7010238;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#322\322]8;;\

                    INFO     list_keywords(lang='en'): 16936 keywords from API.                      ]8;id=7010244;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=7010245;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#335\335]8;;\

list_keywords  local=15,757  api=16,934  jaccard=0.920
  consistency ok


In [52]:
# search_pictograms: IDs returned locally vs via API for a stable keyword
try:
    local_ids = {p['id'] for p in search_pictograms(keyword='water', lang=LANG, max_results=10)['results']}
    with _use_api():
        api_ids = {p['id'] for p in search_pictograms(keyword='water', lang=LANG, max_results=10)['results']}

    overlap = local_ids & api_ids
    print(f'search_pictograms("water")  local={sorted(local_ids)}  api={sorted(api_ids)}')
    print(f'  overlap={sorted(overlap)}  ({len(overlap)}/{max(len(local_ids), len(api_ids))})')
    assert len(overlap) >= 1, 'Expected at least 1 common ID between local and API'
    print('  consistency ok')
except Exception as exc:
    print(f'  SKIP (network error): {exc}')


                    INFO     search_pictograms('water', lang='en'): 8 results from local dataset.    ]8;id=7010250;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=7010251;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#227\227]8;;\

                    INFO     search_pictograms('water', lang='en'): 10 results from API.             ]8;id=7010257;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=7010258;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#240\240]8;;\

search_pictograms("water")  local=[2248, 2816, 6889, 22082, 24354, 24355, 32464, 34173]  api=[2248, 2816, 3172, 3277, 6889, 22082, 24354, 24355, 32464, 34173]
  overlap=[2248, 2816, 6889, 22082, 24354, 24355, 32464, 34173]  (8/10)
  consistency ok


In [53]:
# get_pictogram_metadata: same ID → same core fields
try:
    local_meta = get_pictogram_metadata(pictogram_id=WATER_ID, lang=LANG)
    with _use_api():
        api_meta = get_pictogram_metadata(pictogram_id=WATER_ID, lang=LANG)

    assert local_meta['id'] == api_meta['id'] == WATER_ID
    local_kws_set = {k['keyword'] for k in local_meta.get('keywords', [])}
    api_kws_set   = {k['keyword'] for k in api_meta.get('keywords', [])}
    assert 'water' in local_kws_set and 'water' in api_kws_set, (
        f'"water" missing: local={local_kws_set}  api={api_kws_set}'
    )
    print(f'get_pictogram_metadata(id={WATER_ID}): id matches, "water" kw present in both — ok')
except Exception as exc:
    print(f'  SKIP (network error): {exc}')


                    INFO     get_pictogram_metadata(id=2248): OK from local dataset.                 ]8;id=7010263;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=7010264;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#274\274]8;;\

[05/12/26 10:41:48] INFO     get_pictogram_metadata(id=2248): OK from API.                           ]8;id=7010270;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=7010271;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#292\292]8;;\

get_pictogram_metadata(id=2248): id matches, "water" kw present in both — ok


In [54]:
# search_pictograms_by_synset: same synset → same IDs (or substantial overlap)
try:
    local_syn = {p['id'] for p in search_pictograms_by_synset(synset_id=WATER_SYNSET, lang=LANG)['results']}
    with _use_api():
        api_syn = {p['id'] for p in search_pictograms_by_synset(synset_id=WATER_SYNSET, lang=LANG)['results']}

    overlap_syn = local_syn & api_syn
    ratio = len(overlap_syn) / max(len(local_syn), len(api_syn), 1)
    print(f'search_by_synset("{WATER_SYNSET}")  local={sorted(local_syn)}  api={sorted(api_syn)}')
    print(f'  overlap={sorted(overlap_syn)}  ratio={ratio:.2f}')
    assert WATER_ID in local_syn and WATER_ID in api_syn, (
        f'id={WATER_ID} must be in both local ({local_syn}) and api ({api_syn})'
    )
    print('  consistency ok')
except Exception as exc:
    print(f'  SKIP (network error): {exc}')


                    INFO     search_pictograms_by_synset(synset=07951744-n, lang=en): 3 results from ]8;id=7010276;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=7010277;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#372\372]8;;\
                             local dataset.                                                                        

                    INFO     search_pictograms_by_synset(synset=07951744-n, wn=3.1, lang=en): 3      ]8;id=7010283;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=7010284;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#397\397]8;;\
                             results from API.                                                                     

search_by_synset("07951744-n")  local=[2248, 6889, 32464]  api=[2248, 6889, 32464]
  overlap=[2248, 6889, 32464]  ratio=1.00
  consistency ok


## 8. `resolve_concept`


In [55]:
from agent.resolve import resolve_concept
from agent.resolve import _SPACY_OK

print(f'spaCy available: {_SPACY_OK}')
print(f'kw_set size: {len(kw_set):,}')


[05/12/26 10:41:52] INFO     resolve: spaCy en_core_web_sm loaded — lemmatisation active.             ]8;id=7010291;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/agent/resolve.py\resolve.py]8;;\:]8;id=7010292;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/agent/resolve.py#32\32]8;;\

spaCy available: True
kw_set size: 15,757


In [57]:
# Step 1 — exact match
assert resolve_concept('water', kw_set) == ['water'], 'Exact match failed'
assert resolve_concept('eat', kw_set)   == ['eat'],   'Exact match failed for "eat"'
print('Step 1 (exact match): ok')

# Step 2 — lemma (only when spaCy is available)
if _SPACY_OK:
    if 'go out' in kw_set:
        result = resolve_concept('going out', kw_set)
        assert result == ['go out'], f'Lemma step failed: {result}'
        print('Step 2 (full-phrase lemma "going out" → "go out"): ok')
    else:
        print('Step 2: "go out" not in kw_set — skipping')
else:
    print('Step 2: spaCy not installed — skipping lemma tests')

Step 1 (exact match): ok
Step 2 (full-phrase lemma "going out" → "go out"): ok


In [59]:
# Step 3 — space/hyphen normalisation
tested = 0
for kw in sorted(kw_set):  # sorted for determinism
    if '-' in kw:
        space_form = kw.replace('-', ' ')
        if space_form not in kw_set:
            result = resolve_concept(space_form, kw_set)
            if kw in result:   # only count cases that actually work
                tested += 1
                if tested >= 3:
                    break

if tested > 0:
    print(f'Step 3 (space↔hyphen): tested {tested} case(s) ok')
else:
    print('Step 3 (space↔hyphen): no clean hyphen↔space cases found in index — skipping')

Step 3 (space↔hyphen): tested 3 case(s) ok


In [60]:
# Step 4 — token fallback
# 'wash hands': neither in kw_set as phrase → tokens 'wash' and 'hand' separately
if 'wash hands' not in kw_set:
    result = resolve_concept('wash hands', kw_set)
    # At least one token should be in index
    print(f'resolve_concept("wash hands") → {result}')
    assert isinstance(result, list), 'Must return a list'
    print(f'Step 4 (token fallback): {len(result)} token(s) found')
else:
    print('Step 4: "wash hands" is in kw_set — exact match, fallback not triggered')

# Edge: empty input → empty list
assert resolve_concept('', kw_set) == [], 'Empty input must return []'
# Edge: totally unknown concept → empty list
unknown = resolve_concept('__zzz_nonexistent_xyz__', kw_set)
assert unknown == [], f'Unknown concept must return [], got {unknown}'
print('Edge cases (empty, unknown): ok')


resolve_concept("wash hands") → ['wash', 'hand']
Step 4 (token fallback): 2 token(s) found
Edge cases (empty, unknown): ok


In [61]:
# return_method=True — verify (queries, method) tuple returned by eval path
queries, method = resolve_concept('water', kw_set, return_method=True)
assert isinstance(queries, list) and isinstance(method, str), \
    f'return_method=True must return (list, str), got ({type(queries)}, {type(method)})'
assert method == 'exact', f'"water" should resolve via exact, got {method!r}'
assert queries == ['water'], f'queries mismatch: {queries}'
print(f'return_method=True (exact): queries={queries}  method={method!r}  ok')

# Unknown concept → ([], 'none')
q2, m2 = resolve_concept('__zzz__', kw_set, return_method=True)
assert q2 == [] and m2 == 'none', f'Expected ([], "none"), got ({q2}, {m2!r})'
print(f'return_method=True (none):  queries={q2}  method={m2!r}  ok')

# Verify all labels in RESOLVE_METHODS are strings
from agent.resolve import RESOLVE_METHODS
assert all(isinstance(m, str) for m in RESOLVE_METHODS), 'RESOLVE_METHODS must be all strings'
print(f'RESOLVE_METHODS: {RESOLVE_METHODS}')


return_method=True (exact): queries=['water']  method='exact'  ok
return_method=True (none):  queries=[]  method='none'  ok
RESOLVE_METHODS: ('exact', 'lemma', 'hyphen', 'lemma_alt', 'token', 'none')
